# HALO Harness Optimization

This notebook keeps the existing Databricks agent demo intact, then adds a harness optimization pass with [HALO](https://github.com/context-labs/HALO).

The goal is broader than prompt optimization or skill generation. HALO reviews the same evidence used by skills generation and produces a Codex-ready handoff for improving the whole harness: prompts, tool routing, sufficiency checks, Genie fallback behavior, skills, eval coverage, and deployment guardrails.

Prerequisites:
- Run `00_setup.ipynb` through `09-Evaluation.ipynb` for the existing demo path.
- Run `05-JudgeAlignment.ipynb` so the aligned judge has semantic and episodic memory.
- Run `06-PromptOptimization.ipynb` and `07-AgentSkillsGeneration.ipynb` so optimized prompt and skills artifacts exist.
- Use `gpt-5-4-external` as the model endpoint for HALO. If Databricks OpenAI-compatible routing does not support HALO's Agents SDK calls, this notebook can fall back to direct OpenAI with `OPENAI_API_KEY`.

In [ ]:
import json
from datetime import datetime, timezone

BOOTSTRAP_VOLUME_DIR = "/Volumes/main/at_bat_assistant/agent_skills_gepa/halo"
BOOTSTRAP_STATUS_PATH = f"{BOOTSTRAP_VOLUME_DIR}/run_status.json"
BOOTSTRAP_LOG_PATH = f"{BOOTSTRAP_VOLUME_DIR}/run_log.txt"
payload = {
    "ts": datetime.now(timezone.utc).isoformat(),
    "stage": "bootstrap",
    "message": "Starting HALO dependency install cell",
    "data": {"note": "If this remains the latest status, the notebook is still in %pip install or Python restart."},
}
dbutils.fs.mkdirs(BOOTSTRAP_VOLUME_DIR)
dbutils.fs.put(BOOTSTRAP_STATUS_PATH, json.dumps(payload, indent=2), True)
dbutils.fs.put(BOOTSTRAP_LOG_PATH, json.dumps(payload) + "\n", True)
print(payload)


In [ ]:
%pip install -q --prefer-binary "halo-engine==0.1.10" "mlflow>=3.11.1" "typing_extensions>=4.15.0" "databricks-agents>=1.6.0" openai


In [ ]:
from __future__ import annotations

import json
import os
import re
import time
import traceback
import uuid
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import sys
sys.modules.pop("typing_extensions", None)

import mlflow
from databricks.sdk import WorkspaceClient
from IPython.display import Markdown, display

CONFIG = json.loads(Path("config/atbat_assistant.json").read_text())
CATALOG = CONFIG["workspace"]["catalog"]
SCHEMA = CONFIG["workspace"]["schema"]
EXPERIMENT_ID = CONFIG["mlflow"]["experiment_id"]
PROMPT_NAME = CONFIG["prompt_registry"]["prompt_name"]
ALIGNED_JUDGE_NAME = CONFIG["judges"]["aligned_judge_name"]
SKILLS_VOLUME_PATH = CONFIG["skills"]["gepa_volume_path"]
UC_TOOL_NAMES = CONFIG["tools"]["uc_tool_names"]
HALO_MODEL = os.getenv("HALO_MODEL", CONFIG["llm"].get("endpoint_name", "gpt-5-4-external")).replace("databricks:/", "")
HALO_DIRECT_OPENAI_MODEL = os.getenv("HALO_DIRECT_OPENAI_MODEL", "gpt-5.4-nano")

os.environ.setdefault("MLFLOW_TRACKING_URI", "databricks")
os.environ.setdefault("MLFLOW_REGISTRY_URI", f"databricks-uc://{CATALOG}")
mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri(f"databricks-uc://{CATALOG}")
try:
    mlflow.set_experiment(experiment_id=EXPERIMENT_ID)
except Exception as exc:
    print(f"Skipping active experiment setup; trace search will use explicit experiment location. Reason: {exc}")
w = WorkspaceClient()
ARTIFACT_ROOT = Path("/tmp/atbat_halo")
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
HALO_TRACE_PATH = ARTIFACT_ROOT / "halo_traces.jsonl"
HALO_OUTPUT_JSON = ARTIFACT_ROOT / "halo_output.json"
HALO_HANDOFF_MD = ARTIFACT_ROOT / "codex_handoff.md"
HALO_LOG_PATH = ARTIFACT_ROOT / "run_log.txt"
HALO_STATUS_JSON = ARTIFACT_ROOT / "run_status.json"
HALO_VOLUME_DIR = f"{SKILLS_VOLUME_PATH.rstrip('/')}/halo"
HALO_VOLUME_LOG_PATH = f"{HALO_VOLUME_DIR}/run_log.txt"
HALO_VOLUME_STATUS_PATH = f"{HALO_VOLUME_DIR}/run_status.json"
HALO_LOG_PATH.write_text("", encoding="utf-8")

def log_event(stage: str, message: str, **data: Any) -> None:
    payload = {
        "ts": datetime.now(timezone.utc).isoformat(),
        "stage": stage,
        "message": message,
        "data": data,
    }
    line = json.dumps(payload, default=str, ensure_ascii=False)
    print(f"[{payload['ts']}] {stage}: {message} {data if data else ''}")
    with HALO_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(line + "\n")
    HALO_STATUS_JSON.write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")
    try:
        dbutils.fs.mkdirs(HALO_VOLUME_DIR)
        dbutils.fs.put(HALO_VOLUME_LOG_PATH, HALO_LOG_PATH.read_text(encoding="utf-8"), True)
        dbutils.fs.put(HALO_VOLUME_STATUS_PATH, HALO_STATUS_JSON.read_text(encoding="utf-8"), True)
    except Exception as exc:
        print(f"Could not persist live HALO log: {type(exc).__name__}: {exc}")

print(f"Experiment ID: {EXPERIMENT_ID}")
print(f"HALO Databricks model endpoint: {HALO_MODEL}")
print(f"HALO direct OpenAI fallback model: {HALO_DIRECT_OPENAI_MODEL}")
print(f"Artifact root: {ARTIFACT_ROOT}")
log_event("init", "HALO notebook initialized", experiment_id=EXPERIMENT_ID, halo_model=HALO_MODEL, artifact_root=str(ARTIFACT_ROOT), volume_dir=HALO_VOLUME_DIR)

## Load The Same Evidence Used For Skills Generation

HALO should optimize the harness from the same evidence set used by `07-AgentSkillsGeneration.ipynb`: UC function signatures, optimized prompt, aligned judge memory, evaluated traces, and generated skills.

In [ ]:
def _safe_json(value: Any, max_chars: int = 20000) -> str:
    try:
        text = json.dumps(value, default=str, ensure_ascii=False)
    except Exception:
        text = str(value)
    if len(text) > max_chars:
        return text[:max_chars] + f"... [truncated {len(text) - max_chars} chars]"
    return text


def _read_volume_text(path: str) -> str:
    try:
        return Path(path).read_text()
    except Exception:
        try:
            return dbutils.fs.head(path, 200000)
        except Exception:
            return ""


def _list_volume_files(path: str, *, recursive: bool = False, exclude_prefixes: tuple[str, ...] = ()) -> list[str]:
    def _is_excluded(candidate: str) -> bool:
        return any(candidate.rstrip('/').startswith(prefix.rstrip('/')) for prefix in exclude_prefixes)

    try:
        entries = dbutils.fs.ls(path)
    except Exception:
        return []

    files = []
    for entry in entries:
        entry_path = entry.path
        if _is_excluded(entry_path):
            continue
        if entry_path.endswith('/'):
            if recursive:
                files.extend(_list_volume_files(entry_path, recursive=True, exclude_prefixes=exclude_prefixes))
        else:
            files.append(entry_path)
    return files


def _extract_user_query(trace) -> str:
    try:
        request = trace.data.request
        if isinstance(request, str):
            request = json.loads(request)
        inputs = request.get("input", request.get("inputs", [])) if isinstance(request, dict) else []
        if isinstance(inputs, dict):
            inputs = inputs.get("input", [])
        if isinstance(inputs, list):
            for msg in inputs:
                if isinstance(msg, dict) and msg.get("role") == "user":
                    return str(msg.get("content", ""))
    except Exception:
        pass
    return ""


def _extract_agent_response(trace) -> str:
    try:
        response = trace.data.response
        if isinstance(response, str):
            response = json.loads(response)
        output = response.get("output", []) if isinstance(response, dict) else []
        texts = []
        for item in output if isinstance(output, list) else []:
            if not isinstance(item, dict):
                continue
            for content in item.get("content", []) or []:
                if isinstance(content, dict) and content.get("type") == "output_text":
                    texts.append(str(content.get("text", "")))
        return "\n".join(t for t in texts if t)
    except Exception:
        return ""


def _extract_tool_calls(trace) -> list[dict[str, Any]]:
    try:
        response = trace.data.response
        if isinstance(response, str):
            response = json.loads(response)
        output = response.get("output", []) if isinstance(response, dict) else []
    except Exception:
        output = []
    calls_by_id = {}
    ordered = []
    for item in output if isinstance(output, list) else []:
        if not isinstance(item, dict):
            continue
        if item.get("type") == "function_call":
            call_id = item.get("call_id") or item.get("id") or str(len(ordered))
            entry = {
                "call_id": call_id,
                "name": item.get("name"),
                "arguments": item.get("arguments"),
                "output": None,
            }
            calls_by_id[call_id] = entry
            ordered.append(entry)
        elif item.get("type") == "function_call_output":
            call_id = item.get("call_id")
            if call_id in calls_by_id:
                calls_by_id[call_id]["output"] = item.get("output")
    return ordered


def _assessment_summary(trace) -> list[dict[str, Any]]:
    rows = []
    for assessment in getattr(trace.info, "assessments", []) or []:
        feedback = getattr(assessment, "feedback", None)
        rows.append({
            "name": getattr(assessment, "name", None),
            "source": getattr(getattr(assessment, "source", None), "source_type", None),
            "value": getattr(feedback, "value", None) if feedback else None,
            "rationale": getattr(assessment, "rationale", None),
        })
    return rows

In [ ]:
log_event("evidence", "Loading UC function descriptions", function_count=len(UC_TOOL_NAMES))
# UC function signatures
uc_functions = []
for fn in UC_TOOL_NAMES:
    try:
        rows = spark.sql(f"DESCRIBE FUNCTION EXTENDED {fn}").collect()
        uc_functions.append({"name": fn, "description": "\n".join(str(r[0]) for r in rows)})
    except Exception as exc:
        uc_functions.append({"name": fn, "description": f"ERROR loading signature: {exc}"})
uc_functions_text = "\n\n".join(f"### {f['name']}\n{f['description']}" for f in uc_functions)
log_event("evidence", "Loaded UC function descriptions", loaded=len(uc_functions), errors=sum(1 for f in uc_functions if f['description'].startswith('ERROR')))

log_event("evidence", "Loading optimized production prompt", prompt_name=PROMPT_NAME)
# Optimized prompt
try:
    prompt_obj = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}@production")
    optimized_prompt_text = getattr(prompt_obj, "template", None) or prompt_obj.format()
except Exception as exc:
    optimized_prompt_text = f"ERROR loading production prompt: {exc}"
log_event("evidence", "Loaded optimized prompt", chars=len(optimized_prompt_text), load_failed=optimized_prompt_text.startswith('ERROR'))

log_event("evidence", "Loading aligned judge memory", judge_name=ALIGNED_JUDGE_NAME)
# Aligned judge memory. HALO can still run if the local MLflow wheel does not expose mlflow.genai.scorers.
semantic_memory = []
episodic_count = None
try:
    from mlflow.genai.scorers import get_scorer

    aligned_judge = get_scorer(name=ALIGNED_JUDGE_NAME, experiment_id=EXPERIMENT_ID)
    try:
        _ = aligned_judge(
            inputs={"input": [{"role": "user", "content": "How should a hitter approach a pitcher with runners on base?"}]},
            outputs={"response": "Use count, handedness, pitch mix, and location evidence before making a recommendation."},
        )
    except Exception as exc:
        print(f"Judge warmup raised {type(exc).__name__}; continuing with loaded metadata")

    for g in getattr(aligned_judge, "_semantic_memory", []) or []:
        semantic_memory.append({"guideline": g.guideline_text, "source_trace_ids": g.source_trace_ids})
    episodic_count = len(getattr(aligned_judge, "_episodic_memory", []) or [])
except Exception as exc:
    log_event("evidence", "Skipped direct aligned judge memory load", error_type=type(exc).__name__, error=str(exc)[:1000])
    print("HALO will use trace assessments and evaluation metrics as judge evidence instead.")
log_event("evidence", "Aligned judge memory load complete", semantic_guidelines=len(semantic_memory), episodic_examples=episodic_count)

log_event("evidence", "Loading generated skills from UC volume", skills_volume=SKILLS_VOLUME_PATH)
# Generated skills from UC volume
skill_files = []
for path in _list_volume_files(SKILLS_VOLUME_PATH, recursive=True, exclude_prefixes=(HALO_VOLUME_DIR,)):
    if path.endswith((".md", ".txt", ".json")):
        skill_files.append({"path": path, "content": _read_volume_text(path)[:30000]})
log_event("evidence", "Loaded generated skill artifacts", skill_file_count=len(skill_files), skill_paths=[s['path'] for s in skill_files[:10]])

## Build HALO-Compatible Trace JSONL

HALO expects canonical span JSONL. This cell converts MLflow evaluation traces into one root span plus one span per tool call, preserving user query, response, assessments, tool arguments, and tool outputs.

In [ ]:
def _iso_now() -> str:
    return datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")


def _span(trace_id: str, name: str, attrs: dict[str, Any], parent_span_id: str = "") -> dict[str, Any]:
    sid = uuid.uuid4().hex[:16]
    return {
        "trace_id": trace_id.replace("tr-", "")[:32].ljust(32, "0"),
        "span_id": sid,
        "parent_span_id": parent_span_id,
        "trace_state": "",
        "name": name,
        "kind": "SPAN_KIND_INTERNAL",
        "start_time": _iso_now(),
        "end_time": _iso_now(),
        "status": {"code": "STATUS_CODE_OK", "message": ""},
        "resource": {"attributes": {"service.name": "at-bat-assistant", "project.id": "at-bat-assistant-cais"}},
        "scope": {"name": "mlflow-to-halo", "version": "1"},
        "attributes": attrs,
    }

log_event("traces", "Searching for aligned traces", experiment_id=EXPERIMENT_ID, filter="tag.align = 'use'")
# Prefer the aligned traces if present because these are the examples that drive the harness loop.
traces = mlflow.search_traces(
    locations=[EXPERIMENT_ID],
    filter_string="tag.align = 'use'",
    return_type="list",
)
log_event("traces", "Aligned trace search complete", trace_count=len(traces))
if not traces:
    log_event("traces", "No aligned traces found; falling back to eval-complete traces", filter="tag.eval = 'complete'")
    traces = mlflow.search_traces(
        locations=[EXPERIMENT_ID],
        filter_string="tag.eval = 'complete'",
        return_type="list",
    )
log_event("traces", "Loaded traces for HALO", trace_count=len(traces))

log_event("traces", "Converting MLflow traces to HALO span JSONL")
span_rows = []
trace_summaries = []
for trace in traces:
    tid = trace.info.trace_id
    query = _extract_user_query(trace)
    response = _extract_agent_response(trace)
    tools = _extract_tool_calls(trace)
    assessments = _assessment_summary(trace)
    root = _span(tid, "agent_evaluation_trace", {
        "input.value": query,
        "output.value": response,
        "atbat.trace_id": tid,
        "atbat.user_query": query,
        "atbat.agent_response": response,
        "atbat.assessments": _safe_json(assessments),
        "atbat.tool_call_count": len(tools),
        "atbat.tool_calls": _safe_json(tools),
    })
    span_rows.append(root)
    trace_summaries.append({
        "trace_id": tid,
        "query": query,
        "response_preview": response[:500],
        "tool_count": len(tools),
        "assessments": assessments,
    })
    for i, tool in enumerate(tools, 1):
        status = "STATUS_CODE_ERROR" if str(tool.get("output", "")).lower().startswith("error") else "STATUS_CODE_OK"
        child = _span(tid, f"tool_call_{i}:{tool.get('name')}", {
            "tool.name": tool.get("name"),
            "tool.arguments": tool.get("arguments"),
            "tool.output": str(tool.get("output", ""))[:30000],
            "atbat.trace_id": tid,
        }, parent_span_id=root["span_id"])
        child["status"]["code"] = status
        span_rows.append(child)

with HALO_TRACE_PATH.open("w") as f:
    for row in span_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

log_event("traces", "Wrote HALO span JSONL", span_count=len(span_rows), trace_count=len(trace_summaries), trace_path=str(HALO_TRACE_PATH))
try:
    dbutils.fs.put(f"{HALO_VOLUME_DIR}/halo_traces.in_progress.jsonl", HALO_TRACE_PATH.read_text(encoding="utf-8"), True)
    log_event("traces", "Copied in-progress trace JSONL to UC volume", volume_path=f"{HALO_VOLUME_DIR}/halo_traces.in_progress.jsonl")
except Exception as exc:
    log_event("traces", "Could not copy in-progress trace JSONL", error_type=type(exc).__name__, error=str(exc)[:1000])

## Run HALO

The Databricks endpoint is tried first through OpenAI-compatible routing. If that fails and `OPENAI_API_KEY` is available, the notebook retries against direct OpenAI.

In [ ]:
log_event("halo", "Importing HALO engine modules")
from engine.agents.agent_config import AgentConfig
from engine.engine_config import EngineConfig
from engine.main import run_engine_async
from engine.model_config import ModelConfig
from engine.model_provider_config import ModelProviderConfig
from engine.models.messages import AgentMessage


def _workspace_token() -> str:
    try:
        return dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
    except Exception:
        return w.config.authenticate()


def _make_engine_config(model_name: str, *, provider: ModelProviderConfig | None = None) -> EngineConfig:
    root_turns = int(os.getenv("HALO_ROOT_MAX_TURNS", "8"))
    sub_turns = int(os.getenv("HALO_SUB_MAX_TURNS", "6"))
    max_depth = int(os.getenv("HALO_MAX_DEPTH", "1"))
    max_parallel = int(os.getenv("HALO_MAX_PARALLEL_SUBAGENTS", "2"))
    reasoning_effort = os.getenv("HALO_REASONING_EFFORT", "low")
    root_model = ModelConfig(name=model_name, reasoning_effort=reasoning_effort, maximum_output_tokens=12000)
    sub_model = ModelConfig(name=model_name, reasoning_effort=reasoning_effort, maximum_output_tokens=12000)
    synthesis_model = ModelConfig(name=model_name, reasoning_effort=reasoning_effort, maximum_output_tokens=12000)
    compact_model = ModelConfig(name=model_name, maximum_output_tokens=4000)
    return EngineConfig(
        root_agent=AgentConfig(name="halo-root", model=root_model, maximum_turns=root_turns),
        subagent=AgentConfig(name="halo-sub", model=sub_model, maximum_turns=sub_turns),
        synthesis_model=synthesis_model,
        compaction_model=compact_model,
        model_provider=provider or ModelProviderConfig(),
        maximum_depth=max_depth,
        maximum_parallel_subagents=max_parallel,
    )

log_event("halo", "Building HALO context", trace_count=len(trace_summaries), skill_count=len(skill_files), semantic_guidelines=len(semantic_memory))
halo_context = {
    "uc_functions": uc_functions,
    "optimized_prompt": optimized_prompt_text[:50000],
    "semantic_memory": semantic_memory,
    "episodic_memory_count": episodic_count,
    "skills": skill_files,
    "trace_summaries": trace_summaries,
}

halo_prompt = f"""
You are optimizing the full harness for the Databricks at-bat assistant demo.

Use the trace file as the primary evidence. Also use this harness context:
{_safe_json(halo_context, max_chars=90000)}

Your task:
1. Diagnose recurring harness-level failure modes, not one-off answer mistakes.
2. Rank fixes across prompt, UC tool routing, sufficiency evaluation, Genie fallback, skills, eval data, judge alignment, deployment resources, and observability.
3. Identify which fixes are safe to implement automatically and which need manual review.
4. Produce a Codex-loadable Markdown handoff that another Codex session can read and directly use to suggest prompt and harness changes.
5. Preserve the current demo structure: 00 through 09 remain the existing optimization loop; this HALO pass is an additional whole-harness loop.

The Markdown handoff must use these exact top-level sections:
# Codex Implementation Handoff
## Objective
## Evidence
## Recommended Changes
## File/Notebook Targets
## Patch Plan
## Validation Plan
## Risks and Manual Checks

Reference these notebook targets where relevant:
- notebooks/03_create_agent_definition.ipynb
- notebooks/06-PromptOptimization.ipynb
- notebooks/07-AgentSkillsGeneration.ipynb
- notebooks/08_create_agent_with_skills.ipynb
- notebooks/09-Evaluation.ipynb
- notebooks/10-HALOHarnessOptimization.ipynb

Focus the recommendations on agent prompt changes, tool routing changes, sufficiency/fallback harness changes, Genie fallback instructions, skill-generation inputs, and eval instrumentation. Do not write executable code. Make the handoff concrete enough that Codex can propose patches from it.
"""

log_event("halo", "HALO prompt prepared", prompt_chars=len(halo_prompt), trace_path=str(HALO_TRACE_PATH))

async def _run_halo_with_databricks():
    log_event("halo", "Preparing Databricks model provider", endpoint=HALO_MODEL, base_url=f"{w.config.host.rstrip('/')}/serving-endpoints/")
    provider = ModelProviderConfig(
        base_url=f"{w.config.host.rstrip('/')}/serving-endpoints/",
        api_key=_workspace_token(),
    )
    cfg = _make_engine_config(HALO_MODEL, provider=provider)
    log_event("halo", "Starting HALO run via Databricks", endpoint=HALO_MODEL, reasoning_effort=os.getenv("HALO_REASONING_EFFORT", "low"), root_turns=os.getenv("HALO_ROOT_MAX_TURNS", "8"), sub_turns=os.getenv("HALO_SUB_MAX_TURNS", "6"), max_depth=os.getenv("HALO_MAX_DEPTH", "1"), max_parallel=os.getenv("HALO_MAX_PARALLEL_SUBAGENTS", "2"))
    return await run_engine_async([AgentMessage(role="user", content=halo_prompt)], cfg, HALO_TRACE_PATH, telemetry=False)


async def _run_halo_with_direct_openai():
    log_event("halo", "Preparing direct OpenAI fallback", model=HALO_DIRECT_OPENAI_MODEL)
    if not os.getenv("OPENAI_API_KEY"):
        try:
            openai_secret_scope = os.getenv("OPENAI_SECRET_SCOPE", "<your-secret-scope>")
            os.environ["OPENAI_API_KEY"] = dbutils.secrets.get(scope=openai_secret_scope, key="openai_api_key")
        except Exception:
            pass
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("Databricks HALO route failed and OPENAI_API_KEY is not available for direct OpenAI fallback")
    cfg = _make_engine_config(HALO_DIRECT_OPENAI_MODEL)
    log_event("halo", "Starting HALO run via direct OpenAI", model=HALO_DIRECT_OPENAI_MODEL)
    return await run_engine_async([AgentMessage(role="user", content=halo_prompt)], cfg, HALO_TRACE_PATH, telemetry=False)

start = time.time()
try:
    log_event("halo", "Running HALO through Databricks OpenAI-compatible endpoint")
    halo_items = await _run_halo_with_databricks()
    halo_route = "databricks"
    log_event("halo", "Databricks HALO route completed", output_items=len(halo_items))
except Exception as databricks_exc:
    log_event("halo", "Databricks HALO route failed; trying direct OpenAI fallback", error_type=type(databricks_exc).__name__, error=str(databricks_exc)[:2000])
    print(type(databricks_exc).__name__, str(databricks_exc)[:2000])
    try:
        halo_items = await _run_halo_with_direct_openai()
        halo_route = "direct_openai"
        log_event("halo", "Direct OpenAI HALO route completed", output_items=len(halo_items))
    except Exception as openai_exc:
        halo_route = "failed"
        halo_items = []
        raise RuntimeError(
            "HALO failed through both Databricks and direct OpenAI. "
            f"Databricks error: {databricks_exc}\nDirect OpenAI error: {openai_exc}"
        ) from openai_exc

elapsed = time.time() - start
log_event("halo", "HALO run complete", route=halo_route, elapsed_seconds=round(elapsed, 1), output_items=len(halo_items))

In [ ]:
def _item_to_dict(item):
    try:
        return item.model_dump(mode="json")
    except Exception:
        return {"repr": repr(item)}

halo_payload = {
    "route": halo_route,
    "model": HALO_MODEL if halo_route == "databricks" else HALO_DIRECT_OPENAI_MODEL,
    "elapsed_seconds": elapsed,
    "trace_path": str(HALO_TRACE_PATH),
    "items": [_item_to_dict(item) for item in halo_items],
}
HALO_OUTPUT_JSON.write_text(json.dumps(halo_payload, indent=2, default=str), encoding="utf-8")
log_event("artifacts", "Wrote HALO JSON payload locally", path=str(HALO_OUTPUT_JSON), item_count=len(halo_items))

texts = []
for item in halo_items:
    data = _item_to_dict(item)
    obj = data.get("item", {}) if isinstance(data, dict) else {}
    if isinstance(obj, dict) and obj.get("role") == "assistant":
        content = obj.get("content")
        if isinstance(content, str):
            texts.append(content)
        elif isinstance(content, list):
            for part in content:
                if isinstance(part, dict) and part.get("text"):
                    texts.append(part["text"])

handoff = "\n\n".join(texts).strip()
if not handoff:
    handoff = json.dumps(halo_payload, indent=2, default=str)[:100000]

header = f"""# HALO Harness Optimization Handoff

Generated: {datetime.now(timezone.utc).isoformat()}
Route: {halo_route}
Model: {HALO_MODEL if halo_route == 'databricks' else HALO_DIRECT_OPENAI_MODEL}
Trace count: {len(traces)}
Span count: {len(span_rows)}

"""
HALO_HANDOFF_MD.write_text(header + handoff + "\n", encoding="utf-8")
log_event("artifacts", "Wrote Codex handoff locally", path=str(HALO_HANDOFF_MD), handoff_chars=len(handoff))
display(Markdown(HALO_HANDOFF_MD.read_text()[:20000]))

In [ ]:
# Persist HALO artifacts to the same UC volume family used by skills.
halo_volume_dir = HALO_VOLUME_DIR
try:
    log_event("artifacts", "Persisting final HALO artifacts to UC volume", volume_dir=halo_volume_dir)
    dbutils.fs.mkdirs(halo_volume_dir)
    dbutils.fs.put(f"{halo_volume_dir}/halo_traces.jsonl", HALO_TRACE_PATH.read_text(encoding="utf-8"), True)
    dbutils.fs.put(f"{halo_volume_dir}/halo_output.json", HALO_OUTPUT_JSON.read_text(encoding="utf-8"), True)
    dbutils.fs.put(f"{halo_volume_dir}/codex_handoff.md", HALO_HANDOFF_MD.read_text(encoding="utf-8"), True)
    log_event("artifacts", "Persisted final HALO artifacts to UC volume", volume_dir=halo_volume_dir)
except Exception as exc:
    log_event("artifacts", "Could not persist final HALO artifacts", error_type=type(exc).__name__, error=str(exc)[:1000])

log_event("done", "HALO notebook completed; review codex_handoff.md before implementing changes")

Latest redacted HALO run status (2026-05-18):
- Route: databricks.
- Model endpoint: gpt-5-4-external.
- Evidence converted: 19 traces, 124 spans.
- HALO found recurring harness failures: UC tool 504/timeouts, missing required parameters, Genie EMPTY handling, weak fallback action plans, and insufficient orchestration gating.
- Codex handoff recommended: parameter validation/completion, retry/backoff, EMPTY fallback ladder, prompt fallback contract, sufficiency actionability checks, fallback skill updates, targeted regression cases, and structured failure telemetry.
